# Evaluation metrics for age prediction — a deep-dive tutorial

This notebook explains **every metric** reported by `models/evaluate_test.py` for the aging challenge:

| Metric | Full name |
|--------|-----------|
| **MAE** | Mean Absolute Error |
| **RMSE** | Root Mean Squared Error |
| **R²** | Coefficient of Determination |
| **Pearson r** | Pearson correlation coefficient |
| **Spearman ρ** | Spearman rank correlation |

For each metric you will see:
- The **mathematical formula**
- **Intuition** — what it measures and what it ignores
- **Range and interpretation** — good vs bad values
- **A worked example** on a small toy dataset
- **Demonstration on the real competition data** (when available)

---

## Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display

# Locate project root
NOTEBOOK_DIR = Path.cwd().resolve()
PROJ_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "models").exists() else NOTEBOOK_DIR.parent
print("Project root:", PROJ_ROOT)

# Toy dataset: 10 donors, true vs predicted age
np.random.seed(42)
y_true_toy = np.array([28, 35, 42, 50, 55, 61, 67, 72, 78, 85], dtype=float)
y_pred_toy = y_true_toy + np.array([-3, 5, -8, 2, 10, -4, 6, -1, 9, -5], dtype=float)
toy = pd.DataFrame({"true_age": y_true_toy, "predicted_age": y_pred_toy,
                    "error": y_pred_toy - y_true_toy})
print("\nToy dataset (10 donors):")
display(toy)

---
## 1 — Mean Absolute Error (MAE)

$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |\hat{y}_i - y_i|$$

**Intuition:** The average distance (in years) between each prediction and its true value, ignoring direction. If MAE = 8 you are — on average — 8 years wrong per donor.

**Range:** 0 → ∞; lower is better. 0 means perfect predictions.

**Strengths:**
- Same units as the target (years), making it directly interpretable.
- Treats all errors equally (linear penalty).

**Weaknesses:**
- Does not penalise large errors more than small ones.
- Insensitive to the spread of the target — a MAE of 8 on a 20–80 age range is different from MAE of 8 on a 60–70 range.

In [ ]:
mae = mean_absolute_error(y_true_toy, y_pred_toy)
print("Absolute errors:", np.abs(toy["error"].values))
print(f"MAE = mean({np.abs(toy['error'].values)}) = {mae:.2f} years")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(toy)), np.abs(toy["error"]), color="steelblue", alpha=0.8)
ax.axhline(mae, color="red", linestyle="--", label=f"MAE = {mae:.2f} yrs")
ax.set_xlabel("Donor index")
ax.set_ylabel("Absolute error (years)")
ax.set_title("Per-donor absolute errors")
ax.legend()
plt.tight_layout()
plt.show()

---
## 2 — Root Mean Squared Error (RMSE)

$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2}$$

**Intuition:** Like MAE, but **squares** each error before averaging, then takes the square root to return to the original unit. Squaring gives **disproportionately high penalty to large errors**.

**Range:** 0 → ∞; lower is better. RMSE ≥ MAE always.

**When RMSE >> MAE:** the model is making a few large outlier errors. Focus on those donors.

**Strengths:**
- Sensitive to outliers — useful when large errors are particularly costly.

**Weaknesses:**
- Scale-dependent (still in years), but harder to interpret than MAE.
- Dominated by a small number of large errors, masking typical performance.

In [ ]:
rmse = np.sqrt(mean_squared_error(y_true_toy, y_pred_toy))
squared = toy["error"].values ** 2
print("Squared errors:", squared)
print(f"MSE  = mean(squared errors) = {squared.mean():.2f}")
print(f"RMSE = sqrt(MSE) = {rmse:.2f} years")
print(f"MAE  = {mae:.2f}  →  RMSE - MAE = {rmse - mae:.2f}  (gap caused by large errors)")

# Visualise: add an outlier to see how RMSE responds more strongly than MAE
y_pred_outlier = y_pred_toy.copy()
y_pred_outlier[-1] += 25  # add a 25-year error to last donor
print("\nWith one outlier added (+25 years to donor 10):")
print(f"  MAE  {mae:.2f} → {mean_absolute_error(y_true_toy, y_pred_outlier):.2f}")
print(f"  RMSE {rmse:.2f} → {np.sqrt(mean_squared_error(y_true_toy, y_pred_outlier)):.2f}")
print("RMSE increases much more — it is more sensitive to outlier errors.")

---
## 3 — R² (Coefficient of Determination)

$$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}
     = 1 - \frac{SS_{\text{res}}}{SS_{\text{tot}}}$$

where $\bar{y}$ is the mean true age and $SS_{\text{tot}}$ is the total variance in $y$.

**Intuition:** The fraction of the age variance **explained** by the model.  
A "naive" baseline that always predicts the mean age has R² = 0. A perfect model has R² = 1.

**Range:** −∞ → 1; higher is better. Negative means **worse than predicting the mean**.

**Strengths:**
- Scale-independent — you can compare models trained on different data ranges.

**Weaknesses:**
- Sensitive to the range of ages in the test set. A test set spanning 20–80 makes R² appear higher than one spanning 50–65.
- R² can be high even if systematic bias exists (model always overestimates by 5 years).

In [ ]:
ss_res = np.sum((y_true_toy - y_pred_toy) ** 2)
ss_tot = np.sum((y_true_toy - y_true_toy.mean()) ** 2)
r2 = 1 - ss_res / ss_tot
print(f"SS_residual  = {ss_res:.2f}")
print(f"SS_total     = {ss_tot:.2f}  (variance of true ages × n)")
print(f"R²           = 1 - {ss_res:.2f}/{ss_tot:.2f} = {r2:.3f}")
print(f"Interpretation: the model explains {r2*100:.1f}% of the age variance.")

# Demonstrate different R² scenarios
cases = {
    "Perfect": y_true_toy.copy(),
    "Our model": y_pred_toy.copy(),
    "Mean baseline": np.full_like(y_true_toy, y_true_toy.mean()),
    "Terrible": y_true_toy[::-1].copy(),
}
print("\nR² for different scenarios:")
for name, pred in cases.items():
    print(f"  {name:<18}: R² = {r2_score(y_true_toy, pred):.3f}")

---
## 4 — Pearson correlation (r)

$$r = \frac{\sum (y_i - \bar{y})(\hat{y}_i - \bar{\hat{y}})}
          {\sqrt{\sum(y_i - \bar{y})^2 \cdot \sum(\hat{y}_i - \bar{\hat{y}})^2}}$$

**Intuition:** Measures the **strength and direction of the linear relationship** between predicted and true ages. It is not affected by overall scale or offset.

**Range:** −1 → 1. |r| = 1 means perfect linear fit; r = 0 means no linear relationship.

**Key difference from R²:**
- $r^2 = R^2$ **only** when both variables have zero mean. In general: $R^2 = r^2$ after accounting for intercept.
- A model that **systematically overestimates** by 10 years can still have Pearson r = 1, but R² < 1.

**Weaknesses:**
- Only detects **linear** association. A model predicting $\hat{y} = y^2$ could have low Pearson r despite good ordering.
- Sensitive to outliers.

In [ ]:
r, p = pearsonr(y_true_toy, y_pred_toy)
print(f"Pearson r = {r:.3f}  (p = {p:.4f})")

# Demonstrate: biased model has same r but lower R²
y_biased = y_pred_toy + 15   # every prediction shifted 15 years higher
r_bias, _ = pearsonr(y_true_toy, y_biased)
r2_bias   = r2_score(y_true_toy, y_biased)
print(f"\nBiased model (+15 years to all predictions):")
print(f"  Pearson r = {r_bias:.3f}  (unchanged — Pearson is shift-invariant)")
print(f"  R²        = {r2_bias:.3f}  (much worse — R² penalises bias)")
print(f"  MAE       = {mean_absolute_error(y_true_toy, y_biased):.2f}  (much worse)")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, pred, title in zip(
    axes,
    [y_pred_toy, y_biased],
    [f"Our model  r={r:.2f}  R²={r2:.2f}",
     f"Biased +15 r={r_bias:.2f}  R²={r2_bias:.2f}"],
):
    ax.scatter(y_true_toy, pred, color="steelblue", edgecolors="k", zorder=3)
    lim = (20, 100)
    ax.plot(lim, lim, "r--", label="y = x")
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("True age"); ax.set_ylabel("Predicted age")
    ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()

---
## 5 — Spearman rank correlation (ρ)

$$\rho = r_{\text{rank}(y),\, \text{rank}(\hat{y})}$$

Spearman ρ is simply the Pearson correlation applied to the **ranks** of the values rather than the values themselves.

**Intuition:** Measures whether the **ordering** of donors by predicted age agrees with the true ordering. A model that consistently puts younger donors before older ones gets high ρ, even if the scale is wrong.

**Range:** −1 → 1; higher is better.

**Key differences from Pearson:**
- **Non-parametric** — no assumption of normality or linearity.
- **Robust to outliers** — a single extreme error changes ranks by at most n−1.
- Can detect **monotonic but non-linear** relationships that Pearson would miss.

**In age prediction:** Spearman ρ < Pearson r usually means the model has a non-linear bias (e.g. it compresses predictions towards the mean).

In [ ]:
rho, p_spear = spearmanr(y_true_toy, y_pred_toy)
print(f"Spearman ρ = {rho:.3f}  (p = {p_spear:.4f})")
print(f"Pearson  r = {r:.3f}")

# Add an extreme outlier — Spearman barely changes, Pearson is affected
y_out = y_pred_toy.copy(); y_out[4] = 200   # absurd prediction for donor 5
r_out,   _ = pearsonr(y_true_toy, y_out)
rho_out, _ = spearmanr(y_true_toy, y_out)
print(f"\nWith extreme outlier (donor 5 predicted as 200 yrs):")
print(f"  Pearson  r = {r_out:.3f}  (badly distorted)")
print(f"  Spearman ρ = {rho_out:.3f}  (barely changed)")

# Show ranks
from scipy.stats import rankdata
rank_df = pd.DataFrame({
    "true_age":  y_true_toy,
    "predicted": y_pred_toy,
    "rank_true":  rankdata(y_true_toy).astype(int),
    "rank_pred":  rankdata(y_pred_toy).astype(int),
})
print("\nRanks used by Spearman:")
display(rank_df)

---
## 6 — p-values for Pearson and Spearman

Both `pearsonr` and `spearmanr` return a **p-value** alongside the correlation coefficient.

**What it tests:** H₀ = the true correlation is 0 (no relationship between predictions and true ages).

**Interpretation:** A tiny p-value (e.g. p < 10⁻²⁰) means the observed correlation is extremely unlikely to arise by chance — confirming the model has **real predictive signal**. It does **not** mean the model is accurate; a model with r = 0.3 and 1000 donors also gets p ≈ 0.

**Do not confuse significance with accuracy:** a statistically significant correlation still allows large MAE.

In [ ]:
np.random.seed(0)
n_samples = [10, 50, 105, 500]
print(f"{'n':>6}  {'r':>8}  {'p-value':>14}  note")
for n in n_samples:
    ytrue = np.random.uniform(20, 85, n)
    # Simulate a model with moderate correlation (r ≈ 0.5)
    ypred = ytrue * 0.5 + np.random.normal(0, 12, n) + 25
    rn, pn = pearsonr(ytrue, ypred)
    note = "p < 0.05" if pn < 0.05 else "not significant"
    print(f"{n:>6}  {rn:>8.3f}  {pn:>14.2e}  {note}")
print("\nSame signal strength: larger n → smaller p (more power)")

---
## 7 — All metrics together: a visual guide

This cell generates the **3 most common failure modes** to show what each metric detects.

In [ ]:
np.random.seed(7)
true = np.linspace(25, 80, 60) + np.random.normal(0, 2, 60)

cases = {
    "Good model": true + np.random.normal(0, 5, 60),
    "Systematic bias (+10)": true + 10 + np.random.normal(0, 5, 60),
    "Mean regression (compressed)": true * 0.3 + true.mean() * 0.7 + np.random.normal(0, 3, 60),
    "Noisy (random)": np.random.uniform(25, 80, 60),
}

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
print(f"{'Model':<30} {'MAE':>6} {'RMSE':>6} {'R²':>6} {'r':>6} {'ρ':>6}")

for ax, (name, pred) in zip(axes.ravel(), cases.items()):
    mae_v  = mean_absolute_error(true, pred)
    rmse_v = np.sqrt(mean_squared_error(true, pred))
    r2_v   = r2_score(true, pred)
    r_v, _ = pearsonr(true, pred)
    rho_v, _ = spearmanr(true, pred)
    print(f"{name:<30} {mae_v:>6.2f} {rmse_v:>6.2f} {r2_v:>6.3f} {r_v:>6.3f} {rho_v:>6.3f}")

    ax.scatter(true, pred, alpha=0.7, s=25, color="steelblue", edgecolors="k", linewidths=0.4)
    lim = (15, 95)
    ax.plot(lim, lim, "r--", linewidth=1.5, label="y = x")
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel("True age"); ax.set_ylabel("Predicted age")
    ax.set_title(f"{name}\nMAE={mae_v:.1f}  R²={r2_v:.2f}  r={r_v:.2f}  ρ={rho_v:.2f}")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## 8 — Real competition data

We now apply all five metrics to the **actual test predictions** from the baseline model.  
This requires `test_labels_hidden.csv` (ground-truth ages).

In [ ]:
TRUTH_PATH = PROJ_ROOT / "data_prep" / "output" / "test_labels_hidden.csv"
OUT_BASE   = PROJ_ROOT / "models" / "output"

# Find latest test_predictions.csv
def find_latest_pred(base):
    legacy = base / "test_predictions.csv"
    if legacy.exists():
        return legacy
    subdirs = sorted([d for d in base.iterdir() if d.is_dir()], reverse=True)
    for d in subdirs:
        p = d / "test_predictions.csv"
        if p.exists():
            return p
    return None

pred_path = find_latest_pred(OUT_BASE)
print("Predictions:", pred_path)
print("Truth:      ", TRUTH_PATH)

if not TRUTH_PATH.exists() or pred_path is None:
    print("\nOne or both files missing — skipping real-data section.")
    y_true_real = y_pred_real = None
else:
    pred_df  = pd.read_csv(pred_path)
    truth_df = pd.read_csv(TRUTH_PATH)
    merged = pred_df.merge(truth_df, on="donor_id", how="inner")
    y_true_real = merged["age"].values
    y_pred_real = merged["predicted_age"].values
    print(f"Donors evaluated: {len(merged)}")

In [ ]:
if y_true_real is not None:
    mae_r   = mean_absolute_error(y_true_real, y_pred_real)
    rmse_r  = np.sqrt(mean_squared_error(y_true_real, y_pred_real))
    r2_r    = r2_score(y_true_real, y_pred_real)
    r_r, p_r   = pearsonr(y_true_real, y_pred_real)
    rho_r, p_s = spearmanr(y_true_real, y_pred_real)

    print("Real test set evaluation (n=%d donors)" % len(y_true_real))
    results = pd.DataFrame([
        {"Metric": "MAE (years)",   "Value": f"{mae_r:.3f}",  "Interpretation": f"Off by {mae_r:.1f} yrs on average"},
        {"Metric": "RMSE (years)",  "Value": f"{rmse_r:.3f}", "Interpretation": f"Larger errors penalised; gap to MAE = {rmse_r-mae_r:.2f} yrs"},
        {"Metric": "R²",            "Value": f"{r2_r:.3f}",   "Interpretation": f"{r2_r*100:.1f}% of age variance explained"},
        {"Metric": "Pearson r",     "Value": f"{r_r:.3f}",    "Interpretation": f"p = {p_r:.2e} — strong linear correlation"},
        {"Metric": "Spearman ρ",    "Value": f"{rho_r:.3f}",  "Interpretation": f"p = {p_s:.2e} — strong rank correlation"},
    ])
    display(results)

    errors = y_pred_real - y_true_real
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Scatter
    axes[0].scatter(y_true_real, y_pred_real, alpha=0.7, edgecolors="k", linewidth=0.4, color="steelblue")
    lim = (min(y_true_real.min(), y_pred_real.min()) - 5, max(y_true_real.max(), y_pred_real.max()) + 5)
    axes[0].plot(lim, lim, "r--", label="y = x")
    axes[0].set_xlabel("True age"); axes[0].set_ylabel("Predicted age")
    axes[0].set_title(f"Predicted vs True age\nr={r_r:.3f}  R²={r2_r:.3f}")
    axes[0].legend()

    # Error distribution
    axes[1].hist(errors, bins=20, color="steelblue", edgecolor="k", alpha=0.8)
    axes[1].axvline(0, color="red", linestyle="--")
    axes[1].axvline(errors.mean(), color="orange", linestyle="--", label=f"mean={errors.mean():.1f}")
    axes[1].set_xlabel("Prediction error (yrs)"); axes[1].set_ylabel("Count")
    axes[1].set_title(f"Error distribution\nMAE={mae_r:.2f}  RMSE={rmse_r:.2f}")
    axes[1].legend()

    # Error vs true age
    axes[2].scatter(y_true_real, errors, alpha=0.7, edgecolors="k", linewidth=0.4, color="darkorange")
    axes[2].axhline(0, color="red", linestyle="--")
    axes[2].set_xlabel("True age"); axes[2].set_ylabel("Error (predicted − true)")
    axes[2].set_title("Residuals vs true age\n(pattern = systematic bias)")

    plt.tight_layout()
    plt.show()
else:
    print("Skipped — no ground-truth file available.")

---
## 9 — Quick reference

| Metric | Formula | Unit | Range | Better |
|--------|---------|------|-------|--------|
| **MAE** | mean\|error\| | years | 0 → ∞ | lower |
| **RMSE** | √mean(error²) | years | 0 → ∞ | lower |
| **R²** | 1 − SS_res/SS_tot | — | −∞ → 1 | higher |
| **Pearson r** | linear correlation | — | −1 → 1 | \|r\| → 1 |
| **Spearman ρ** | rank correlation | — | −1 → 1 | \|ρ\| → 1 |

### Diagnosis cheat-sheet

| Symptom | Likely cause |
|---------|-------------|
| RMSE >> MAE | A few large outlier errors |
| High Pearson, low R² | Systematic bias (scale/offset off) |
| High Pearson, low Spearman | Non-linear mapping between true and predicted ages |
| High R²/r, tiny p-values, but bad MAE | Large test-set age range inflating R²/r |
| R² < 0 | Worse than predicting the mean — model has failed |

### Competition context

Our baseline (pseudobulk Random Forest) achieved on the **test set**:
- MAE ≈ 8.7 years — typically off by less than a decade
- R² ≈ 0.61 — 61% of age variance explained
- Pearson r ≈ 0.83 — strong linear correlation (p < 10⁻²⁶)

Competitors should aim to beat these numbers, especially MAE and R².